<div align="center">
  <h1>Fe(SH)<sub>4</sub><sup>−</sup> spin state energetics using Quiche and Cuda-Q</h1>
  <p>
    <img src="https://img.shields.io/badge/Python-3.11-3776AB?style=flat&logo=python" alt="Python">
    <img src="https://img.shields.io/badge/CUDA-76B900?style=flat&logo=nvidia" alt="CUDA-Q">
  </p>
  <i>This tutorial is part of a collaboration between <a href="https://quantummotion.com/">Quantum Motion</a> and NVIDIA.</i>
</div>

---
This is a demo implementation showing how a full quantum chemical pipeline could integrate with quantum computing, by computing the spin state energetics of Fe(SH)<sub>4</sub><sup>&minus;</sup> in a minimal 5 electron in 5 orbital active space. For the simulation workflow, we show the integration between:

<table style="border-collapse: collapse; border: none;">
  <tbody>
    <tr>
      <td style="border: none; vertical-align: top; padding: 6px 16px 6px 0;"><b>ORCA</b></td>
      <td style="border: none; vertical-align: top; padding: 6px 0;">An industry-standard quantum chemistry package, used here to extract the active-space integrals for the different spin states.</td>
    </tr>
    <tr>
      <td style="border: none; vertical-align: top; padding: 6px 16px 6px 0;"><b>MPSCircuits.jl</b></td>
      <td style="border: none; vertical-align: top; padding: 6px 0;">A Julia package by <a href="https://quantummotion.com/">Quantum Motion</a> for state-of-the-art state preparation based on matrix product states.</td>
    </tr>
    <tr>
      <td style="border: none; vertical-align: top; padding: 6px 16px 6px 0;"><b>QUICHE</b></td>
      <td style="border: none; vertical-align: top; padding: 6px 0;">A software package by <a href="https://quantummotion.com/">Quantum Motion</a> that provides an interface between quantum computing and chemistry software, for both resource estimation and simulation.</td>
    </tr>
    <tr>
      <td style="border: none; vertical-align: top; padding: 6px 16px 6px 0;"><b>CUDA-Q</b> and <b>CUDA-Q Algorithms</b></td>
      <td style="border: none; vertical-align: top; padding: 6px 0;">NVIDIA's open-source platform for hybrid quantum-classical programming, with a library of ready-made algorithm building blocks, used here to build the qubit Hamiltonians and run GPU-accelerated QPE simulations.</td>
    </tr>
  </tbody>
</table>

## The problem: spin state energetics

The identity of the ground spin state is crucial for understanding reactivity, electrochemistry and photochemistry. Furthermore, the energy differences between the ground spin state and states of other multiplicity are needed to predict, for example, spectroscopy. Iron-sulfur systems are an especially delicate target for these so-called spin state energetics, as the different spin states have been reported to be within sub-miliHartree of each other for certain clusters. Different methods, as well as the inclusion or omission of dynamic correlation effects, can also result in different orderings of the spin ladder. 

Classically, this is a hard problem as the only feasible option for large-scale transition metal complex calculations is density functional theory (DFT). The results from DFT calculations, however, depend heavily on the functional and the strong correlation effects present in such systems cause difficulties.

Because each spin state has its own, distinct Hamiltonian, a calculation with the goal of predicting any of the above properties only becomes useful when it is targeting the correct state.

<p align="center">
  <img src="img/Fe(SH)4-.png" alt="Fe(SH)4- structure" width="400">
</p>

In this dome, we choose Fe(SH)<sub>4</sub><sup>&minus;</sup> as our target system. It functions as a minimal example of an Fe-S cluster, but has its own chemical relevance as it is a minimal model of the iron site of rubredoxin, the simplest natural iron–sulfur protein [1]. 

<table style="border-collapse: collapse; border: none; margin: 0 auto; text-align: center;">
  <thead>
    <tr>
      <th style="border: none; padding: 6px 16px;"></th>
      <th style="border: none; padding: 6px 16px;">Sextet<br><span style="font-weight: normal;">S = 5/2 (high spin)</span></th>
      <th style="border: none; padding: 6px 16px;">Quartet<br><span style="font-weight: normal;">S = 3/2 (intermediate spin)</span></th>
      <th style="border: none; padding: 6px 16px;">Doublet<br><span style="font-weight: normal;">S = 1/2 (low spin)</span></th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: none; padding: 10px 16px; text-align: right; white-space: nowrap;">t<sub>2</sub> (upper)</td>
      <td style="border: none; padding: 10px 16px;">
        <span style="display:inline-block;width:34px;margin:0 4px;border-bottom:2px solid currentColor;line-height:1.5;">↑</span><span style="display:inline-block;width:34px;margin:0 4px;border-bottom:2px solid currentColor;line-height:1.5;">↑</span><span style="display:inline-block;width:34px;margin:0 4px;border-bottom:2px solid currentColor;line-height:1.5;">↑</span>
      </td>
      <td style="border: none; padding: 10px 16px;">
        <span style="display:inline-block;width:34px;margin:0 4px;border-bottom:2px solid currentColor;line-height:1.5;">↑</span><span style="display:inline-block;width:34px;margin:0 4px;border-bottom:2px solid currentColor;line-height:1.5;">↑</span><span style="display:inline-block;width:34px;margin:0 4px;border-bottom:2px solid currentColor;line-height:1.5;">&nbsp;</span>
      </td>
      <td style="border: none; padding: 10px 16px;">
        <span style="display:inline-block;width:34px;margin:0 4px;border-bottom:2px solid currentColor;line-height:1.5;">↑</span><span style="display:inline-block;width:34px;margin:0 4px;border-bottom:2px solid currentColor;line-height:1.5;">&nbsp;</span><span style="display:inline-block;width:34px;margin:0 4px;border-bottom:2px solid currentColor;line-height:1.5;">&nbsp;</span>
      </td>
    </tr>
    <tr>
      <td style="border: none; padding: 10px 16px; text-align: right; white-space: nowrap;">e (lower)</td>
      <td style="border: none; padding: 10px 16px;">
        <span style="display:inline-block;width:34px;margin:0 4px;border-bottom:2px solid currentColor;line-height:1.5;">↑</span><span style="display:inline-block;width:34px;margin:0 4px;border-bottom:2px solid currentColor;line-height:1.5;">↑</span>
      </td>
      <td style="border: none; padding: 10px 16px;">
        <span style="display:inline-block;width:34px;margin:0 4px;border-bottom:2px solid currentColor;line-height:1.5;">↑↓</span><span style="display:inline-block;width:34px;margin:0 4px;border-bottom:2px solid currentColor;line-height:1.5;">↑</span>
      </td>
      <td style="border: none; padding: 10px 16px;">
        <span style="display:inline-block;width:34px;margin:0 4px;border-bottom:2px solid currentColor;line-height:1.5;">↑↓</span><span style="display:inline-block;width:34px;margin:0 4px;border-bottom:2px solid currentColor;line-height:1.5;">↑↓</span>
      </td>
    </tr>
    <tr>
      <td style="border: none; padding: 10px 16px; text-align: right; white-space: nowrap;">unpaired e<sup>&minus;</sup></td>
      <td style="border: none; padding: 10px 16px;">5</td>
      <td style="border: none; padding: 10px 16px;">3</td>
      <td style="border: none; padding: 10px 16px;">1</td>
    </tr>
  </tbody>
</table>

We will examine the doublet, quartet and sextet states of Fe(SH)<sub>4</sub><sup>&minus;</sup> and build an active space of 5 electrons in 5 orbitals, choosing the 3d orbitals of the iron centre to build our active space. As the multiplicity of these states *increases*, the amount of Salter determinants in the wavefunction *decreases*. Using

$$
N_{\mathrm{det}} = \binom{n_{\mathrm{orb}}}{N_\alpha}\binom{n_{\mathrm{orb}}}{N_\beta}
= \frac{n_{\mathrm{orb}}!}{N_\alpha!\,(n_{\mathrm{orb}}-N_\alpha)!}\cdot\frac{n_{\mathrm{orb}}!}{N_\beta!\,(n_{\mathrm{orb}}-N_\beta)!} \thinspace ,
$$

we find

<table style="border-collapse: collapse; border: none; margin: 0 auto; text-align: center;">
  <thead>
    <tr>
      <th style="border: none; border-bottom: 1px solid currentColor; padding: 6px 16px;">Spin state</th>
      <th style="border: none; border-bottom: 1px solid currentColor; padding: 6px 16px;"><i>S</i></th>
      <th style="border: none; border-bottom: 1px solid currentColor; padding: 6px 16px;"><i>N</i><sub>&alpha;</sub></th>
      <th style="border: none; border-bottom: 1px solid currentColor; padding: 6px 16px;"><i>N</i><sub>&beta;</sub></th>
      <th style="border: none; border-bottom: 1px solid currentColor; padding: 6px 16px;">Slater determinants</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: none; padding: 6px 16px;">Sextet</td>
      <td style="border: none; padding: 6px 16px;">5/2</td>
      <td style="border: none; padding: 6px 16px;">5</td>
      <td style="border: none; padding: 6px 16px;">0</td>
      <td style="border: none; padding: 6px 16px;">C(5,5) &times; C(5,0) = <b>1</b></td>
    </tr>
    <tr>
      <td style="border: none; padding: 6px 16px;">Quartet</td>
      <td style="border: none; padding: 6px 16px;">3/2</td>
      <td style="border: none; padding: 6px 16px;">4</td>
      <td style="border: none; padding: 6px 16px;">1</td>
      <td style="border: none; padding: 6px 16px;">C(5,4) &times; C(5,1) = <b>25</b></td>
    </tr>
    <tr>
      <td style="border: none; padding: 6px 16px;">Doublet</td>
      <td style="border: none; padding: 6px 16px;">1/2</td>
      <td style="border: none; padding: 6px 16px;">3</td>
      <td style="border: none; padding: 6px 16px;">2</td>
      <td style="border: none; padding: 6px 16px;">C(5,3) &times; C(5,2) = <b>100</b></td>
    </tr>
  </tbody>
</table>

### References
1. [Bair & Goddard, J. Am. Chem. Soc. 100, 5669 (1978)](https://researchportal.ip-paris.fr/en/publications/theoretical-studies-of-the-ground-and-excited-states-of-a-model-o/)

# Part 0: Setup
---

## a) Paths, hardware and imports

<div style="background-color: #fff3cd; color: #856404; border-left: 5px solid #ffeeba; padding: 15px; margin-top: 15px; margin-bottom: 15px; border-radius: 4px;">
  <strong>⚠️ Paths:</strong> Paths might need to be adjusted depending on where local software is installed.
</div>

In [1]:
import os

# ORCA.
# orca_dir = os.path.expanduser("~/Software/orca-7.0.0-f.beta.4_linux_x86-64/bin")
orca_dir = os.path.expanduser("~/Software/orca_6_1_1_macosx_arm64_openmpi411")
os.environ["PATH"] = orca_dir + os.pathsep + os.environ["PATH"]

# Open MPI 4.1.8, installed in the virtual environment with `pip install openmpi==4.1.8`.
mpi_dir = os.path.expanduser("~/.venvs/quiche-cudaq")
os.environ["OPI_MPI"] = mpi_dir
os.environ["PATH"] = mpi_dir + "/bin" + os.pathsep + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = mpi_dir + "/lib" + os.pathsep + os.environ.get("LD_LIBRARY_PATH", "")

# Allow 4 MPI processes inside a cluster (PBS) job.
os.environ["OMPI_MCA_rmaps_base_oversubscribe"] = "1"

# Threads for CUDA-Q's CPU backend: one per physical core (4 on a g4dn.2xlarge), as in the batch jobs.
# Must be set before cudaq is imported.
os.environ["OMP_NUM_THREADS"] = "4"

<div style="background-color: #fff3cd; color: #856404; border-left: 5px solid #ffeeba; padding: 15px; margin-top: 15px; margin-bottom: 15px; border-radius: 4px;">
  <strong>⚠️ Hardware Notice:</strong> CUDA-Q kernels in this notebook run on a GPU by default, if a GPU is detected. If there are no GPUs found, the demo automatically falls back to CPU usage. On CPU, keep the error budget coarse at first: runtime grows quickly with the number of phase qubits.
</div>

In [2]:
import ctypes
import cudaq

# On a machine without a GPU, CUDA-Q runs on its CPU simulator and the GPU set-up below is skipped.
gpu_available = cudaq.num_available_gpus() > 0
if gpu_available:
    # Pre-load the GPU libraries CUDA-Q needs, so that its GPU backend finds them (fixes the libcustatevec error).
    for lib in os.environ["CUDAQ_DYNLIBS"].split(":"):
        ctypes.CDLL(lib, mode=ctypes.RTLD_GLOBAL)
    cudaq.set_target("nvidia", option="fp32")

In [3]:
import shutil
import time
from pathlib import Path

import pandas as pd

## b) System and calculation settings

Here we set the general settings that will be used throughout our calculations.

- `multiplicities` to select the three different spin states
- `nel` and `norb` define the number of electrons and orbitals in the active space
- `ncores` and `memory` are the **openMPI** settings to parallelise the ORCA calculations across the available CPU cores.
- The `Path`s defined are where the output will be stored. 

In [4]:
# Spin states: 2 (low spin), 4 (intermediate spin) and 6 (high spin).
multiplicities = [2, 4, 6]

# Active space: the five Fe 3d orbitals and their five electrons.
nel, norb = 5, 5

# Parallelisation: one MPI process per physical core, memory in MB per process.
ncores = 4
memory = 6000

# Directories: the geometry is in FeSH4/, each calculation gets its own subdirectory.
workdir = Path.cwd() / "FeSH4"
rohf_dir = workdir / "rohf"
rohf_dir.mkdir(parents=True, exist_ok=True)

# The FCIDUMP files and the JSON summary go to demo/orca output/.
output_dir = Path.cwd() / "orca_output"
output_dir.mkdir(exist_ok=True)
orca_results_file = output_dir / "orca_results.json"

## c) General workflow of the notebook

The notebook is set up in a way that it only calculates what is missing. 

- If there are no output files yet, everything will be run from scratch. In this case, it is highly recommended to use `GPU` infrastructure to get the most out of the calculations in a reasonable time.
- After running the notebook at least once, the output is saved in the directories defined above. If it finds the output, the calculations will be skipped and the output will be loaded in instead.

# Part 1: Quantum Chemistry calculations with ORCA
---

We use [ORCA](https://orcaforum.kofo.mpg.de/) to generate an FCIDUMP file for each spin state. An FCIDUMP file is a standard text format that stores the one- and two-electron integrals of a chosen active space, together with the core energy. It fully defines the electronic Hamiltonian that the quantum algorithm needs to diagonalise.

## a) High-spin ROHF

To generate an initial orthonormal basis for the calculations we run a restricted open-shell Hartree-Fock (ROHF) calculation, which provides starting orbitals for the subsequent active space calculations. For simplicity, we are not using geometry optimised structures for each spin state, so we run this calculation only for the high spin state. The 5 singly occupied orbitals are the ones we will select for the active space.

In [5]:
from opi.core import Calculator
from opi.input.blocks import BlockCasscf, BlockOutput
from opi.input.simple_keywords import BasisSet, Scf, ShellType
from opi.input.structures import Structure

if (rohf_dir / "FeSH4_rohf.gbw").exists():
    print("ROHF orbitals already calculated:", rohf_dir / "FeSH4_rohf.gbw")
else:
    # Set up the ORCA calculator.
    calc = Calculator(basename="FeSH4_rohf", working_dir=rohf_dir)
    calc.structure = Structure.from_xyz(workdir / "fesh4.xyz")
    calc.structure.charge, calc.structure.multiplicity = -1, 6
    calc.input.ncores = ncores
    calc.input.memory = memory
    calc.input.add_simple_keywords(ShellType.ROHF, BasisSet.DEF2_SVP)

    # Run the calculation.
    calc.write_input()
    calc.run()

    # Extract the output.
    output = calc.get_output()
    if not output.terminated_normally():
        raise RuntimeError("ORCA did not terminate normally.")
    output.parse()
    print("ROHF energy:", output.get_final_energy())

ROHF orbitals already calculated: /Users/xeno.devriendt/Documents/Git/Forks/quiche/demo/FeSH4/rohf/FeSH4_rohf.gbw


## b) CASSCF(5e,5o) and FCIDUMP for each spin state

For each spin state, we run a complete active space - self consistent field (CASSCF) calculation on the five Fe 3d electrons and orbitals then writes out the integrals for the doublet, quartet and sextet states. The CASSCF energies also serve as the exact reference for the active space, against which the QPE results are later compared.

For a larger system or active space, we would not run a full CASSCF calculation, but rather extract the integrals directly from a Hartree-Fock level calculation. Here, however, we choose to run CASSCF to use it as a reference calculation for later.

In [6]:
results = pd.read_json(orca_results_file).to_dict("records") if orca_results_file.exists() else []

for multiplicity in [m for m in multiplicities if not (output_dir / f"FeSH4_mult{m}.fcidump").exists()]:
    casscf_dir = workdir / f"casscf_mult{multiplicity}"
    casscf_dir.mkdir(parents=True, exist_ok=True)

    # Set up the ORCA calculator, starting from the ROHF orbitals.
    calc = Calculator(basename="FeSH4", working_dir=casscf_dir)
    calc.structure = Structure.from_xyz(workdir / "fesh4.xyz")
    calc.structure.charge, calc.structure.multiplicity = -1, multiplicity
    calc.input.ncores = ncores
    calc.input.memory = memory
    calc.input.add_simple_keywords(ShellType.ROHF, BasisSet.DEF2_SVP, Scf.MOREAD)
    shutil.copy(rohf_dir / "FeSH4_rohf.gbw", casscf_dir / "FeSH4_rohf.gbw")
    calc.input.moinp = casscf_dir / "FeSH4_rohf.gbw"

    # Set up the active space.
    calc.input.add_blocks(BlockCasscf(nel=nel, norb=norb, mult=multiplicity))

    # Set up saving the integrals.
    output_block = BlockOutput()
    output_block.add_option("DumpActInts", "True")
    calc.input.add_blocks(output_block)

    # Run the calculation and time it.
    calc.write_input()
    start = time.perf_counter()
    calc.run()
    walltime = time.perf_counter() - start

    # Extract the output.
    output = calc.get_output()
    if not output.terminated_normally():
        raise RuntimeError(f"ORCA did not terminate normally for multiplicity {multiplicity}.")
    output.parse()

    # Save the FCIDUMP file and the results for this spin state.
    shutil.copy(casscf_dir / "FeSH4.fcidump", output_dir / f"FeSH4_mult{multiplicity}.fcidump")
    results.append({
        "multiplicity": multiplicity,
        "spin": (multiplicity - 1) / 2,
        "energy": output.get_final_energy(),
        "walltime_s": walltime,
    })
    print(f"Multiplicity {multiplicity}: E = {output.get_final_energy():.8f} Ha in {walltime:.0f} s")

In [7]:
# Save the summary of all spin states.
df = pd.DataFrame(results)
df.to_json(orca_results_file, orient="records", indent=2)

# CASSCF over the whole active space, i.e. FCI within the active orbitals: the reference energies.
fci_energy = {r["multiplicity"]: r["energy"] for r in results}

df

,multiplicity,spin,energy,walltime_s
0,2,0.5,-2854.213658,822.842428
1,4,1.5,-2854.269899,314.308871
2,6,2.5,-2854.378194,85.519572


The CASSCF calculations already show us what the correct spin ladder for this system, in this active space looks like. The respective runtimes for each state also confirm what was stated earlier: the *higher* the spin multiplicity, the *lower* the number of determinants in the CI expansion.

# Part 2: DMRG guide states (pre-calculated)
---

The DMRG guide states were computed once with MPSCircuits.jl and saved in `orca_output/`: for each spin state a circuit (`FeSH4_mult{m}_dmrg_circuit.json`) and a summary with the DMRG energy, ⟨S²⟩ and circuit fidelity (`FeSH4_mult{m}_dmrg_info.json`). 

In this particular demo, we only load the initial states. If you want to learn more about initial state preparation using DMRG and matrix product states, have a look [here](https://quantummotion.com/state-preparation-for-quantum-chemistry-with-quantum-motion-and-nvidia-cuda-q/)!

In [8]:
import json

from statePrep.cudaq_circuit_loading import load_circuit_from_json, write_and_import_kernel

# The pre-calculated DMRG guide states: a summary (energy, <S^2>, fidelity) and a circuit per spin state.
dmrg_info, dmrg = {}, {}
for multiplicity in multiplicities:
    dmrg_info[multiplicity] = json.loads((output_dir / f"FeSH4_mult{multiplicity}_dmrg_info.json").read_text())

    # Load the circuit as a CUDA-Q kernel; a different kernel name per spin state,
    # so the three kernels don't overwrite each other.
    circuit_path = output_dir / f"FeSH4_mult{multiplicity}_dmrg_circuit.json"
    dmrg[multiplicity] = write_and_import_kernel(
        load_circuit_from_json(circuit_path), kernel_name=f"dmrg_mult{multiplicity}", module_name=f"dmrg_mult{multiplicity}"
    )


def make_dmrg_state(dmrg_kernel):
    """Wrap a DMRG circuit in a kernel that allocates its own qubits (one kernel per spin state)."""

    @cudaq.kernel
    def dmrg_state():
        q = cudaq.qvector(2 * norb)  # one qubit per spin orbital
        dmrg_kernel(q)

    return dmrg_state

# Part 3: Quantum Phase Estimation: using Quiche and Cuda-Q Algorithms
---

With the output from **ORCA** and our initial guide states all set up, we can now move on to creating the quantum phase estimation (QPE) kernels and running them. We will use **Quiche** to read in and parse the output from our initial **ORCA** calculations, load in the qubit Hamiltonians from **Cuda-Q** and set up the `QPESpec` object using **Quiche**. Finally, we will use **Cuda-Q** kernels to run the QPE algorithm. 

### Quantum phase estimation in a nutshell

Quantum phase estimation (QPE) reads out the energy of a Hamiltonian $H$ from the phase it imprints on a quantum state. If $H|\psi_k\rangle = E_k|\psi_k\rangle$, then time evolution acts on each eigenstate as a pure phase:

$$
U = e^{-iHt}, \qquad U|\psi_k\rangle = e^{-iE_k t}|\psi_k\rangle .
$$

**The circuit.** A register of $n$ *phase qubits* is put into an equal superposition. It then controls the evolution $U^{2^j}$ on the *system register*, with qubit $j$ driving $U^{2^j}$, so the phase of $U$ is written onto the phase register in binary. An inverse quantum Fourier transform converts this into a computational-basis state, and measuring the register gives an integer that encodes the energy.

**Guide state.** The system register starts in a guide state $|\psi\rangle = \sum_k c_k |\psi_k\rangle$. A measurement returns the energy $E_k$ with probability $|c_k|^2$, so the ground-state energy is found with probability $|c_0|^2$. This is why a high-overlap guide state matters, and why we prepare DMRG states instead of a single Hartree&ndash;Fock determinant.

**Energy resolution.** With evolution time $t = \pi/\lambda$, where $\lambda$ is the one-norm of the Pauli coefficients of $H$ (an upper bound on its spectral range), the whole spectrum $[-\lambda, \lambda]$ maps onto one full turn of the phase. The readout grid then has spacing

$$
\Delta E = \frac{2\lambda}{2^{n}},
$$

so every extra phase qubit halves the energy resolution, at the cost of doubling the number of controlled evolutions.

**Trotterisation.** $e^{-iHt}$ is not implemented exactly. $H$ is a sum of Pauli terms, so the evolution is approximated by repeating a product of short Pauli rotations (a Trotter&ndash;Suzuki formula). More Trotter steps or a higher order reduce this error but deepen the circuit. QUICHE splits the total energy error budget between the estimation error (the phase grid) and the simulation error (Trotter) and chooses the number of phase qubits, the evolution time, and the Trotter order and steps accordingly.

**Practical details in this demo.**

- **Constant shift.** The identity term $c_0$ (nuclear repulsion plus frozen core) is thousands of Hartree, so it would dominate $\lambda$ ($\approx$ 2864 Ha, compared with $\approx$ 11 Ha without it) and make the grid far too coarse. We therefore run QPE on $H - c_0$ and add $c_0$ back to the measured energy.
- **Sampling.** The circuit is sampled many times, and the median of the measured energies is taken as the estimate.
- **Reference.** The estimate is compared with the CASSCF energy (full CI in the active space), and we increase the number of phase qubits until the error falls below chemical accuracy, 1.6 mHa.

In [9]:
from cudaq_algorithms import chemistry
from quiche.chemistry import HartreeFockState, get_jw_state
from quiche.io import fcidump

# Jordan-Wigner qubit Hamiltonian for each spin state, from its FCIDUMP.
h_cq = {}
for multiplicity in multiplicities:
    sq_ham = fcidump.read(str(output_dir / f"FeSH4_mult{multiplicity}.fcidump"))
    h_cq[multiplicity] = chemistry.qubit_hamiltonian(sq_ham.one_body, sq_ham.two_body, scalar_offset=sq_ham.core_energy)

In [10]:
import numpy as np

import quiche
from quiche.core import ElectronicHamiltonian, Errors, PhaseEstimation, Simulation
from quiche.cudaq import measured_circuit_kernel, sample_post_processor, textbook_qpe_circuit
from quiche.dispatch import QPESpec

# QPE evolves H - c0, and c0 is added back to every measured energy.
# c0 (nuclear repulsion + frozen core) is thousands of Ha. Left in, it dominates the 1-norm that sets the
# evolution time t = pi/lambda, and the readout grid becomes far too coarse to see the active-space energy.
h_qpe, c0 = {}, {}
for multiplicity in multiplicities:
    h = quiche.PauliSum.from_cudaq(h_cq[multiplicity])
    c0[multiplicity], h_qpe[multiplicity] = h.split_identity()
    print(f"Multiplicity {multiplicity}: {h.num_words} Pauli terms, lambda = {h.lam:.1f} Ha, without c0: {h_qpe[multiplicity].lam:.1f} Ha")

Multiplicity 2: 875 Pauli terms, lambda = 2864.4 Ha, without c0: 11.4 Ha
Multiplicity 4: 743 Pauli terms, lambda = 2864.4 Ha, without c0: 11.3 Ha
Multiplicity 6: 487 Pauli terms, lambda = 2864.2 Ha, without c0: 11.1 Ha


In [11]:
def get_qpe(multiplicity, error, reps=None):
    # Split the budget between the Trotter error and the QPE error; QPESpec picks the
    # QPE ancillas and the Trotter time/order/steps from it.
    qpe = QPESpec(
        hamiltonian=ElectronicHamiltonian(
            num_electrons=nel, mapping=quiche.Mapping.JordanWigner, paulis=h_qpe[multiplicity]),
        state_prep=None,  # the DMRG kernel is injected below instead
        algorithm=PhaseEstimation.Textbook,
        simulation=Simulation.Trotter,
        error_budget=Errors(
            estimation=error / 2,
            simulation=error / 2,
            rotations=error / 2,
            state_prep=error / 2,
            overlap=1.0,
        ),
    )
    # Default trotter step budgetting if no custom is given.
    if reps is None:
        reps = qpe.reps

    kernel, work_qubits, decode = textbook_qpe_circuit(
        h_qpe[multiplicity], Simulation.Trotter, qpe.num_qpe_ancillas, qpe.time, reps, qpe.order,
        n_qubits=qpe.num_data,
    )
    circuit = measured_circuit_kernel(
        kernel, work_qubits, qpe.num_qpe_ancillas, state_prep=dmrg[multiplicity], num_data=qpe.num_data
    )
    post = sample_post_processor(decode, 1)
    return qpe, circuit, post

In [12]:
# Different output file if we limit trotter budgeting. None = default quiche qpe.
trotter_reps = 1
results_file = output_dir / f"qpe_trotter_reps_{trotter_reps}_results.json"

shots = 1000
chem_acc = 1.6e-3  # Ha

# Upper limit on the phase register per backend; each extra phase qubit roughly doubles the runtime.
max_phase_qubits = {"GPU": 13, "CPU": 5}

qpe_results = pd.read_json(results_file).to_dict("records") if (results_file).exists() else []
done = {(r["backend"], r["multiplicity"], r["error_budget"]): r for r in qpe_results}

# Without a GPU, only the CPU runs (those not in the results file yet).
for backend in (["GPU", "CPU"] if gpu_available else ["CPU"]):
    # Select the backend: the GPU in single precision, the CPU in double precision.
    if backend == "GPU":
        cudaq.set_target("nvidia", option="fp32")
    else:
        cudaq.set_target("qpp-cpu")
    cudaq.sample(make_dmrg_state(dmrg[multiplicities[0]]), shots_count=1)  # start the backend up outside the timing

    for multiplicity in multiplicities:
        # Halve the error budget (one extra phase qubit each time) until we reach chemical accuracy.
        error = 2.0
        while True:
            if (backend, multiplicity, error) in done:
                result = done[(backend, multiplicity, error)]  # already run earlier: reuse it
            else:
                qpe, circuit, post = get_qpe(multiplicity, error, reps=trotter_reps)
                if qpe.num_qpe_ancillas > max_phase_qubits[backend]:
                    print(f"{backend} | mult {multiplicity} | stopped: would need more than {max_phase_qubits[backend]} phase qubits")
                    break

                # Simulate and time it.
                start = time.perf_counter()
                samples = cudaq.sample(circuit, shots_count=shots)
                walltime = time.perf_counter() - start

                energy = float(np.median(post(samples))) + c0[multiplicity]
                result = {
                    "backend": backend,
                    "multiplicity": multiplicity,
                    "error_budget": error,
                    "phase_qubits": qpe.num_qpe_ancillas,
                    "total_qubits": qpe.num_qpe_ancillas + qpe.num_data,
                    "trotter_reps": trotter_reps or qpe.reps,
                    "energy": energy,
                    "abs_error": abs(energy - fci_energy[multiplicity]),
                    "walltime_s": walltime,
                }
                # Save after every run, so an interrupted run loses nothing.
                qpe_results.append(result)
                pd.DataFrame(qpe_results).to_json(results_file, orient="records", indent=2)
                print(f"{backend} | mult {multiplicity} | {result['phase_qubits']} phase qubits | "
                      f"|E - E_FCI| = {result['abs_error']:.2e} Ha | {walltime:.1f} s")

            if result["abs_error"] <= chem_acc:
                print(f"{backend} | mult {multiplicity} | chemical accuracy reached with {result['phase_qubits']} phase qubits")
                break
            error /= 2

CPU | mult 2 | stopped: would need more than 5 phase qubits
CPU | mult 4 | stopped: would need more than 5 phase qubits
CPU | mult 6 | stopped: would need more than 5 phase qubits


# Part 4: Analysis
---

Now that the results have been calculated, we are interested in several things. 

- How many estimation qubits are needed in order to get the absolute energies to within chemical accuracy?
- Can we correctly model the spin ladder for Fe(SH)<sub>4</sub><sup>&minus;</sup>?
- How does the runtime scale with respect to the number of estimation qubits?
- How long does our calculation run on a single NVIDIA T4 GPU compared to the available CPU backend, when we want to reach chemical accuracy?

We will go through these questions 1-by-1!

In [13]:
from visualisation import plot_energy_analysis, plot_runtime_analysis, plot_runtime_for_chemical_accuracy

## a) Analysis of the absolute and relative energies

In [14]:
plot_energy_analysis(results_file, orca_results_file)

Phase qubits,Spin ordering,ΔE quartet − sextet (mHa),ΔE doublet − quartet (mHa)
CASSCF,,108.3,56.2
4,wrong,-47.9,80.5
5,wrong,-47.9,80.5
6,correct,304.1,84.7
7,wrong,-45.1,82.6
8,correct,129.5,83.6
9,correct,129.5,39.1
10,correct,107.5,61.1
11,correct,107.6,50.1
12,correct,107.6,61.2


We need **thirteen** estimation qubits to reach chemical accuracy. For all three spin states, the error |E<sub>QPE</sub> &minus; E<sub>FCI</sub>| first drops below the 1.6 mHa threshold with 13 phase qubits (0.29, 0.31 and 1.0 mHa for the doublet, quartet and sextet). With 12 phase qubits all three are still just above it (2.5, 2.4 and 1.7 mHa). The convergence is not monotonic, and the error sometimes plateaus over several qubits (the sextet sits near 7 &mu;Ha for 8&ndash;10 qubits). This is expected, because the energy is read out on a discrete grid whose spacing halves with every extra phase qubit, and each point is a median over a finite number of shots.

We also look at the relative spin ladder. At the CASSCF(5e,5o) level the sextet is the ground state, with the quartet 108.3 mHa above it and the doublet a further 56.2 mHa above the quartet. QPE recovers the correct ordering from 8 phase qubits onwards. It gets it wrong with 4, 5 and 7 qubits, and the single correct case at 6 is a coincidence, since the gap there is far from the reference. With 13 phase qubits the two gaps are 107.6 and 55.6 mHa, within 1 mHa of CASSCF. Because the gaps are large, the ordering is much easier to get than the absolute energies.

## b) Analysis of the runtime

In [15]:
plot_runtime_analysis(results_file)

On the GPU, the runtime barely changes at small register sizes (a few seconds for 4&ndash;8 qubits) and then grows by roughly a factor of 4 per extra qubit. For the doublet it goes from 57 s at 10 qubits to 288 s, 1072 s, 4159 s and 16418 s at 11&ndash;14 qubits. Each extra phase qubit doubles the size of the state vector and also doubles the number of controlled time-evolution steps the circuit has to apply. The sextet is consistently the fastest, at about 56% of the doublet's runtime at 14 qubits. This matches the ratio of Pauli terms in the Hamiltonians (487 versus 875), as expected since the cost of each Trotter step scales with the number of terms.

In [16]:
plot_runtime_for_chemical_accuracy(results_file)

On a single NVIDIA T4, reaching chemical accuracy (13 phase qubits) takes 1.2 hours for the doublet, 58.8 minutes for the quartet and 38.6 minutes for the sextet. The CPU backend (4 cores, double precision) could only be run up to 5 phase qubits, where it was already about 800&ndash;1700 times slower than the GPU. Extrapolating its roughly 4-fold growth per qubit gives about 7.3, 6.1 and 4.1 years for the three spin states, a speed-up of around 5 &times; 10<sup>4</sup> for the GPU.

---

<div align="center">
  <h2>🎉 Demo Complete!</h2>
</div>

🏆 **What you've accomplished in this demo:**

- Generated FCIDUMP files with ORCA (ROHF followed by CASSCF(5e,5o)) for the doublet, quartet and sextet states of Fe(SH)<sub>4</sub><sup>&minus;</sup>
- Loaded DMRG guide states as CUDA-Q kernels
- Specified textbook QPE with an energy error budget in QUICHE and simulated it in CUDA-Q
- Reached chemical accuracy (1.6 mHa) for all three spin states with 13 phase qubits
- Recovered the correct spin ladder (sextet &lt; quartet &lt; doublet), with both gaps within 1 mHa of CASSCF
- Compared a single NVIDIA T4 GPU against the CPU backend: about 40&ndash;70 minutes per spin state on the GPU, versus years extrapolated for the CPU

---